# 📈 Stock Market Data Analyzer — Exploratory Data Analysis
**Tickers:** AAPL · TSLA · GOOGL · MSFT  
**Period:** 2020 – 2025  
> ⚠️ This notebook is for **educational purposes only** and does not constitute financial advice.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.data_fetcher import fetch_all
from src.data_cleaner import clean_all
from src.analyzer import run_full_analysis
from src.config import TICKERS, TICKER_COLORS

plt.style.use('dark_background')
print('Libraries loaded ✅')

## 1. Data Fetching & Cleaning

In [ ]:
raw    = fetch_all()
clean  = clean_all(raw)
results = run_full_analysis(clean)
summary = results['_summary']
print('Data ready ✅')

In [ ]:
# Preview AAPL data
results['AAPL'].tail(10)

## 2. Summary Statistics

In [ ]:
summary

In [ ]:
# Descriptive stats for each ticker
for ticker in TICKERS:
    print(f'\n── {ticker} ──')
    print(results[ticker][['Close','Daily_Return','Volatility_30d']].describe().round(3))

## 3. Closing Price Comparison

In [ ]:
fig = go.Figure()
for t in TICKERS:
    df = results[t]
    fig.add_trace(go.Scatter(x=df.index, y=df['Close'], name=t,
                              line=dict(color=TICKER_COLORS[t], width=1.8)))

fig.update_layout(
    title='Closing Prices — 2020 to 2025',
    paper_bgcolor='#0D1117', plot_bgcolor='#0D1117',
    font=dict(color='white'), height=450,
    xaxis=dict(gridcolor='#1E2D3D'), yaxis=dict(gridcolor='#1E2D3D'),
)
fig.show()

## 4. Cumulative Returns

In [ ]:
fig = go.Figure()
for t in TICKERS:
    df = results[t]
    fig.add_trace(go.Scatter(x=df.index, y=df['Cumulative_Return'], name=t,
                              line=dict(color=TICKER_COLORS[t], width=1.8)))
fig.update_layout(
    title='Cumulative Return (%) — 2020 to 2025',
    paper_bgcolor='#0D1117', plot_bgcolor='#0D1117',
    font=dict(color='white'), height=450,
    xaxis=dict(gridcolor='#1E2D3D'), yaxis=dict(gridcolor='#1E2D3D'),
)
fig.show()

## 5. Moving Averages — AAPL

In [ ]:
df = results['AAPL']
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['Close'],   name='Close',   line=dict(color='#00D4FF', width=1.8)))
fig.add_trace(go.Scatter(x=df.index, y=df['SMA_20'],  name='SMA 20',  line=dict(color='#FFD700', width=1.2, dash='dot')))
fig.add_trace(go.Scatter(x=df.index, y=df['SMA_50'],  name='SMA 50',  line=dict(color='#FF6B6B', width=1.2, dash='dot')))
fig.add_trace(go.Scatter(x=df.index, y=df['SMA_200'], name='SMA 200', line=dict(color='#90EE90', width=1.4)))
fig.update_layout(
    title='AAPL — Close + Moving Averages',
    paper_bgcolor='#0D1117', plot_bgcolor='#0D1117',
    font=dict(color='white'), height=450,
    xaxis=dict(gridcolor='#1E2D3D'), yaxis=dict(gridcolor='#1E2D3D'),
)
fig.show()

## 6. RSI — TSLA

In [ ]:
df = results['TSLA']
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['RSI_14'], name='RSI 14', line=dict(color='#FF4B4B', width=1.5)))
fig.add_hline(y=70, line=dict(color='red',   width=1, dash='dash'))
fig.add_hline(y=30, line=dict(color='green', width=1, dash='dash'))
fig.update_yaxes(range=[0, 100])
fig.update_layout(
    title='TSLA — RSI 14',
    paper_bgcolor='#0D1117', plot_bgcolor='#0D1117',
    font=dict(color='white'), height=380,
    xaxis=dict(gridcolor='#1E2D3D'), yaxis=dict(gridcolor='#1E2D3D'),
)
fig.show()

## 7. Correlation Heatmap

In [ ]:
import pandas as pd
closes = pd.DataFrame({t: results[t]['Close'] for t in TICKERS})
corr   = closes.pct_change().corr()

fig, ax = plt.subplots(figsize=(7, 5), facecolor='#0D1117')
ax.set_facecolor('#0D1117')
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, annot_kws={'color':'white'})
ax.set_title('Return Correlation Matrix', color='white')
ax.tick_params(colors='white')
plt.tight_layout()
plt.show()

## 8. Daily Return Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), facecolor='#0D1117')
axes = axes.flatten()
colors = ['#00D4FF', '#FF4B4B', '#FFD700', '#7FFF00']

for i, (t, c) in enumerate(zip(TICKERS, colors)):
    ret = results[t]['Daily_Return'].dropna()
    axes[i].set_facecolor('#0D1117')
    axes[i].hist(ret, bins=80, color=c, alpha=0.75)
    axes[i].axvline(ret.mean(), color='white', linestyle='--', linewidth=1)
    axes[i].set_title(f'{t} Daily Return Distribution', color='white')
    axes[i].tick_params(colors='white')
    for spine in axes[i].spines.values():
        spine.set_edgecolor('#1E2D3D')

plt.tight_layout()
plt.savefig('../outputs/charts/nb_return_distributions.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.show()

---
> ⚠️ **Disclaimer:** This analysis is for **educational purposes only**. It does not constitute financial advice. Always consult a qualified financial advisor before making investment decisions.